# 02 — Data Cleaning

**Author:** Kuthab Ibrahim  
**Goal:** Apply the cleaning rules in `src/cleaner.py` to the raw nba.com player-stats CSVs and the Kaggle injury CSV, then validate that:

1. Player names are normalized (lowercase, no accents, no Jr./III suffixes).
2. Injury dates are mapped to NBA seasons (`2015-16` style).
3. Missing PER / efficiency stats are filled with season medians.
4. Traded players appear once per (player, season) — preferring the `TOT` aggregate row.
5. Key columns have **zero nulls** afterwards.

Each section shows the *before vs. after* counts and the rules that fired.

In [1]:
import os, sys, glob
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from src import cleaner

RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)
pd.set_option('display.max_columns', 30)

## 1. Load the raw datasets

There are 24 per-season player-stats CSVs (2000–2023) plus one Kaggle injury CSV. We concatenate the player-stats files first.

In [2]:
raw_stats = cleaner.load_raw_player_stats(RAW_DIR)
print('raw stats shape :', raw_stats.shape)
print('seasons covered :', raw_stats["season"].nunique())
raw_stats.head(3)

raw stats shape : (11659, 68)
seasons covered : 24


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,...,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT,season
0,920,A.C. Green,A.C.,1610612748,MIA,37.0,82,50,32,0.610,1411.818333,144,324,0.444,0,...,273,265,232,271,263,220,39,212,95,218,126,26,223,1,2000-01
1,2062,A.J. Guyton,A.J.,1610612741,CHI,23.0,33,6,27,0.182,628.538333,78,192,0.406,27,...,221,326,336,311,57,92,112,292,378,317,224,26,314,1,2000-01
2,243,Aaron McKie,Aaron,1610612755,PHI,28.0,76,51,25,0.671,2390.726667,338,714,0.473,53,...,29,28,30,271,376,316,112,87,20,84,77,4,81,1,2000-01


In [3]:
raw_injuries = pd.read_csv(os.path.join(RAW_DIR, 'nba_injuries.csv'))
print('raw injuries shape:', raw_injuries.shape)
raw_injuries.head(3)

raw injuries shape: (33771, 5)


,Date,Team,Acquired,Relinquished,Notes
0,2000-01-03,Bulls,NaN,Fred Hoiberg,placed on IL with strained left quadricep tendon
1,2000-01-03,Bulls,Randy Brown,NaN,activated from IL
2,2000-01-03,Warriors,Daron Blaylock / Mookie Blaylock,NaN,activated from IL


## 2. Rule — normalize player names

Lowercase, strip accents, drop punctuation and Jr./III/IV suffixes. The Kaggle file lists names like " Marc Gasol" with leading whitespace and accented characters; the nba.com feed has clean names. Both feeds need the same normalized key for joining later.

In [4]:
samples = ['LeBron James Jr.', '  Luka Dončić ', "Shaq O'Neal III", 'JaVale McGee']
pd.DataFrame({'raw': samples,
              'normalized': [cleaner.normalize_player_name(s) for s in samples]})

,raw,normalized
0,LeBron James Jr.,lebron james
1,Luka Dončić,luka doncic
2,Shaq O'Neal III,shaq oneal
3,JaVale McGee,javale mcgee


## 3. Rule — standardize injury dates to NBA season strings

The Kaggle file ships an ISO date column. We attach a `season` label using NBA convention (October–June): a Nov 2015 transaction belongs to season `2015-16`.

In [5]:
demo = pd.DataFrame({'Date': ['2015-11-12','2016-04-01','2016-08-19','2018-02-09']})
demo = cleaner.standardize_injury_dates(demo)
demo

,Date,season
0,2015-11-12,2015-16
1,2016-04-01,2015-16
2,2016-08-19,2016-17
3,2018-02-09,2017-18


## 4. Rule — fill missing efficiency stats with season medians

PER, FG%, and minutes-per-game can be `NaN` for players who played a handful of minutes (division by zero in the source). Filling with the season median keeps cross-era comparisons honest while preventing those rows from being dropped.

In [6]:
demo = pd.DataFrame({
    'season': ['2010-11']*5 + ['2011-12']*5,
    'fg_pct': [0.45, np.nan, 0.50, 0.41, 0.48, 0.47, 0.44, np.nan, 0.49, 0.46]
})
filled = cleaner.fill_missing_with_season_median(demo, columns=['fg_pct'])
demo.assign(filled_fg_pct=filled['fg_pct'])

,season,fg_pct,filled_fg_pct
0,2010-11,0.45,0.450
1,2010-11,NaN,0.465
2,2010-11,0.50,0.500
3,2010-11,0.41,0.410
4,2010-11,0.48,0.480
5,2011-12,0.47,0.470
6,2011-12,0.44,0.440
7,2011-12,NaN,0.465
8,2011-12,0.49,0.490
9,2011-12,0.46,0.460


## 5. Rule — deduplicate traded players

When a player is traded mid-season, basketball-reference adds a `TOT` row that aggregates both stints. nba.com does not, but the same logic applies: keep one row per `(player_id, season)`. We prefer `TOT` if present, otherwise the row with the most games played (which is the player's primary team for that season).

In [7]:
demo = pd.DataFrame({
    'PLAYER_ID': [1,1,1, 2,2],
    'season':    ['2018-19']*3 + ['2018-19']*2,
    'TEAM_ABBREVIATION':['LAL','HOU','TOT', 'TOR','DAL'],
    'GP': [30, 25, 55, 60, 22]
})
cleaner.dedupe_traded_players(demo)

,PLAYER_ID,season,TEAM_ABBREVIATION,GP
0,1,2018-19,TOT,55
1,2,2018-19,TOR,60


## 6. Run the full cleaning pipeline

`cleaner.cleaner()` runs every rule above end-to-end and writes the result to `data/processed/`.

In [8]:
result = cleaner.cleaner(raw_dir=RAW_DIR, processed_dir=PROCESSED_DIR, write_files=True)
stats_clean = result['player_stats']
inj_clean   = result['injuries']
print(f'cleaned player_stats: {len(stats_clean):>6,} rows  |  cols: {stats_clean.shape[1]}')
print(f'cleaned injuries    : {len(inj_clean):>6,} rows  |  cols: {inj_clean.shape[1]}')

cleaned player_stats: 11,659 rows  |  cols: 15
cleaned injuries    : 18,950 rows  |  cols: 10


## 7. Before-vs-after summary

How many rows were dropped or fixed by each rule?

In [9]:
raw_n   = len(raw_stats)
after_zero_gp = int((raw_stats['GP'].fillna(0) > 0).sum())
after_dedupe  = len(stats_clean)

summary = pd.DataFrame({
    'stage': ['raw', 'after drop GP=0 / null player_id', 'after dedupe traded players', 'final clean'],
    'rows':  [raw_n, after_zero_gp, after_dedupe, after_dedupe],
    'delta': [0, raw_n - after_zero_gp, after_zero_gp - after_dedupe, 0],
})
summary

,stage,rows,delta
0,raw,11659,0
1,after drop GP=0 / null player_id,11659,0
2,after dedupe traded players,11659,0
3,final clean,11659,0


In [10]:
raw_inj_n = len(raw_injuries)
no_relinquished = int(raw_injuries['Relinquished'].isna().sum())
exploded = len(inj_clean)
matched_player_id = int(inj_clean['player_id'].notna().sum())

pd.DataFrame({
    'stage': [
        'raw injury transactions',
        'rows with NULL Relinquished (activations only)',
        'exploded multi-player rows -> 1 row per player',
        'rows where player_id resolved via name join'
    ],
    'count': [raw_inj_n, no_relinquished, exploded, matched_player_id]
})

,stage,count
0,raw injury transactions,33771
1,rows with NULL Relinquished (activations only),15868
2,exploded multi-player rows -> 1 row per player,18950
3,rows where player_id resolved via name join,17416


## 8. Validate — no nulls in key columns

In [11]:
key_cols = ['player_id', 'season', 'team', 'games_played', 'minutes_per_game',
            'points_per_game', 'field_goal_percentage']
null_report = stats_clean[key_cols].isna().sum().to_frame('null_count')
null_report['null_pct'] = (null_report['null_count'] / len(stats_clean) * 100).round(3)
null_report

,null_count,null_pct
player_id,0,0.0
season,0,0.0
team,0,0.0
games_played,0,0.0
minutes_per_game,0,0.0
points_per_game,0,0.0
field_goal_percentage,0,0.0


In [12]:
dupes = stats_clean.duplicated(subset=['player_id','season']).sum()
print(f'(player_id, season) duplicates after cleaning: {dupes}')
assert dupes == 0, 'dedupe rule failed'

(player_id, season) duplicates after cleaning: 0


In [13]:
inj_keys = ['player_name_norm', 'season', 'date_of_injury', 'injury_type', 'games_missed']
inj_clean[inj_keys].isna().sum().to_frame('null_count').assign(
    null_pct=lambda d: (d['null_count']/len(inj_clean)*100).round(3))

,null_count,null_pct
player_name_norm,0,0.0
season,0,0.0
date_of_injury,0,0.0
injury_type,0,0.0
games_missed,0,0.0


## 9. Preview the cleaned tables

In [14]:
stats_clean.head(5)

,player_id,player_name,player_name_norm,season,team,position,age,games_played,minutes_per_game,points_per_game,assists_per_game,rebounds_per_game,blocks_per_game,steals_per_game,field_goal_percentage
0,3,Grant Long,grant long,2000-01,VAN,UNK,35.0,66,22.897601,6.000000,1.257576,4.151515,0.227273,1.090909,0.439
1,3,Grant Long,grant long,2001-02,MEM,UNK,36.0,66,28.304571,6.318182,2.060606,3.500000,0.181818,0.954545,0.426
2,3,Grant Long,grant long,2002-03,BOS,UNK,37.0,41,11.944593,1.756098,0.609756,2.024390,0.024390,0.219512,0.386
3,15,Eric Piatkowski,eric piatkowski,2000-01,LAC,UNK,30.0,81,26.475576,10.617284,1.185185,2.975309,0.234568,0.567901,0.433
4,15,Eric Piatkowski,eric piatkowski,2001-02,LAC,UNK,31.0,71,24.178216,8.816901,1.577465,2.591549,0.169014,0.577465,0.439


In [15]:
inj_clean.head(5)

,player_id,player_name,player_name_norm,team,season,date_of_injury,injury_type,required_surgery,games_missed,notes
0,<NA>,(James) Mike Scott,(james) mike scott,Hawks,2012-13,2013-03-01,other,0,2,placed on IL
1,<NA>,(James) Mike Scott,(james) mike scott,Hawks,2014-15,2014-11-12,back,0,1,placed on IL with back injury
2,<NA>,(James) Mike Scott,(james) mike scott,Hawks,2014-15,2014-12-10,illness,0,1,placed on IL with flu
3,<NA>,(James) Mike Scott,(james) mike scott,Hawks,2014-15,2015-03-13,foot,0,9,placed on IL with fractured left big toe
4,<NA>,(James) Mike Scott,(james) mike scott,Hawks,2014-15,2015-04-15,back,0,2,placed on IL with back injury


**Outputs written:** `data/processed/player_stats_clean.csv` and `data/processed/injuries_clean.csv`. The next notebook (`04_eda_visualization.ipynb`) loads those files directly to skip the heavy lifting.